<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/07-pandas/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7: Solutions

Try the exercises first. The notes after each solution explain why it's written that way, and the pandas trap each exercise is built around: guessed types, missing values in conditions, side effects on the caller's data, leakage, misaligned columns and duplicated rows.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from io import StringIO


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _penguins_small():
    return pd.DataFrame({
        "species": ["Adelie", "Adelie", "Gentoo", "Gentoo", "Chinstrap", "Adelie", "Gentoo"],
        "island": ["Dream", "Torgersen", "Biscoe", "Biscoe", "Dream", "Biscoe", "Biscoe"],
        "bill_length_mm": [39.0, 40.0, 47.0, 50.0, 49.0, 38.0, 46.0],
        "bill_depth_mm": [19.5, 20.0, 15.0, 16.0, 17.5, 19.0, 14.0],
        "flipper_length_mm": [190.0, 195.0, 215.0, 225.0, 196.0, 185.0, 210.0],
        "body_mass_g": [3700.0, 4000.0, 5000.0, 5600.0, 3600.0, 3500.0, np.nan],
        "sex": ["male", "female", "male", np.nan, "female", "male", "female"],
    })


def check_load_scores(f):
    text = "id,date,score\n007,2025-01-03,0.9\n012,2025-02-10,?\n101,2025-02-11,-\n230,2025-03-01,0.75\n"

    def load():
        path = Path(tempfile.mkdtemp()) / "scores.csv"
        path.write_text(text, encoding="utf-8")
        return f(path)

    _scenarios("load_scores", [
        ("ids keep their leading zeros", lambda: load()["id"].tolist(), ["007", "012", "101", "230"],
         "pass dtype={'id': str} to read_csv"),
        ("dates are real dates", lambda: pd.api.types.is_datetime64_any_dtype(load()["date"]), True, "parse_dates=['date']"),
        ("'?' and '-' are missing", lambda: load()["score"].isna().tolist(), [False, True, True, False],
         "na_values=['?', '-']"),
        ("score is a float column", lambda: load()["score"].dtype.kind, "f", None),
    ])


def check_missing_summary(f):
    df = pd.DataFrame({"a": [1, None, None, 4], "b": [None, 2, 3, 4], "c": [1, 2, 3, 4], "d": [None, None, 1, 2]})
    expected = pd.DataFrame(
        {"n_missing": [2, 2, 1, 0], "pct_missing": [0.5, 0.5, 0.25, 0.0]},
        index=["a", "d", "b", "c"],
    )
    _scenarios("missing_summary", [
        ("counts, fractions and order", lambda: f(df), expected,
         "sort by n_missing (high to low), then by column name; the index holds the column names"),
        ("nothing missing", lambda: f(pd.DataFrame({"x": [1, 2]}))["n_missing"].tolist(), [0], None),
    ])


def check_filter_and_sort(f):
    df = _penguins_small()

    def unchanged():
        d = _penguins_small()
        f(d, "Adelie", 3600)
        return d.equals(_penguins_small())

    expected = pd.DataFrame({
        "species": ["Adelie", "Adelie"],
        "island": ["Torgersen", "Dream"],
        "body_mass_g": [4000.0, 3700.0],
    })
    _scenarios("filter_and_sort", [
        ("filter, keep three columns, sort, reset the index", lambda: f(df, "Adelie", 3600), expected,
         "the result should have a fresh index 0, 1, 2, ... (reset_index(drop=True))"),
        ("nothing matches", lambda: len(f(df, "Gentoo", 9000)), 0, None),
        ("missing masses are excluded", lambda: f(df, "Gentoo", 0)["body_mass_g"].tolist(), [5600.0, 5000.0], None),
        ("input is not modified", unchanged, True, None),
    ])


def check_add_features(f):
    def unchanged():
        d = _penguins_small()
        f(d)
        return list(d.columns) == list(_penguins_small().columns)

    _scenarios("add_features", [
        ("bill_ratio", lambda: f(_penguins_small())["bill_ratio"].round(4).tolist()[:3], [2.0, 2.0, 3.1333], None),
        ("mass_kg", lambda: f(_penguins_small())["mass_kg"].tolist()[:3], [3.7, 4.0, 5.0], None),
        ("missing mass stays missing", lambda: bool(f(_penguins_small())["mass_kg"].isna().iloc[-1]), True, None),
        ("all original columns are kept", lambda: list(f(_penguins_small()).columns)[:7], list(_penguins_small().columns), None),
        ("input is not modified", unchanged, True, "use assign, or copy() first"),
    ])


def check_species_table(f):
    expected = pd.DataFrame(
        {
            "n": [3, 3, 1],
            "mean_mass": [5300.0, 3733.3, 3600.0],
            "mean_flipper": [216.7, 190.0, 196.0],
            "pct_male": [0.5, 2 / 3, 0.0],
        },
        index=pd.Index(["Gentoo", "Adelie", "Chinstrap"], name="species"),
    )
    _scenarios("species_table", [
        ("named aggregation, rounding and order", lambda: f(_penguins_small()), expected,
         "n counts rows; means skip NaN; pct_male is the share of 'male' among penguins whose sex is known; sort by mean_mass, high to low"),
    ])


def check_fill_by_group(f):
    train = pd.DataFrame({"g": ["a", "a", "a", "b", "b", "c"], "x": [1.0, 3.0, np.nan, 10.0, 20.0, np.nan]})
    test = pd.DataFrame({"g": ["a", "b", "c", "d", "a"], "x": [np.nan, np.nan, np.nan, np.nan, 100.0]})

    def unchanged():
        tr, te = train.copy(), test.copy()
        f(tr, te, "x", "g")
        return tr.equals(train) and te.equals(test)

    _scenarios("fill_by_group", [
        ("training data: group medians", lambda: f(train, test, "x", "g")[0]["x"].tolist(), [1.0, 3.0, 2.0, 10.0, 20.0, 6.5],
         "compute the medians from train only; group c has no values at all in train, so it gets the overall training median (6.5)"),
        ("test data uses training medians", lambda: f(train, test, "x", "g")[1]["x"].tolist(), [2.0, 15.0, 6.5, 6.5, 100.0],
         "compute the medians from train only; groups with no training values, or unseen groups like 'd', get the overall training median"),
        ("inputs are not modified", unchanged, True, None),
    ])


def check_zscore_within_group(f):
    df = pd.DataFrame({"g": ["a", "a", "b", "b", "b", "c"], "x": [1.0, 3.0, 10.0, 10.0, 40.0, 7.0]}, index=[10, 11, 12, 13, 14, 15])
    s = np.sqrt(200)
    expected = pd.Series([-1.0, 1.0, -10 / s, -10 / s, 20 / s, 0.0], index=[10, 11, 12, 13, 14, 15], name="x")
    _scenarios("zscore_within_group", [
        ("z-score within each group, same index as the input", lambda: f(df, "x", "g"), expected,
         "use groupby(...).transform('mean') and transform('std', ddof=0); a group with std 0 gives 0"),
    ])


def check_one_hot_aligned(f):
    train = pd.DataFrame({"island": ["Biscoe", "Dream", "Biscoe"], "mass": [1.0, 2.0, 3.0]})
    test = pd.DataFrame({"island": ["Torgersen", "Dream"], "mass": [4.0, 5.0]})
    expected_train = pd.DataFrame({"mass": [1.0, 2.0, 3.0], "island_Biscoe": [1, 0, 1], "island_Dream": [0, 1, 0]})
    expected_test = pd.DataFrame({"mass": [4.0, 5.0], "island_Biscoe": [0, 0], "island_Dream": [0, 1]})
    _scenarios("one_hot_aligned", [
        ("training encoding", lambda: f(train, test, ["island"])[0], expected_train, "pd.get_dummies(..., columns=columns, dtype=int)"),
        ("test has exactly the training columns", lambda: f(train, test, ["island"])[1], expected_test,
         "reindex(columns=train_encoded.columns, fill_value=0): unseen categories are dropped, missing ones are 0"),
    ])


def check_top_n_per_group(f):
    df = pd.DataFrame({
        "g": ["b", "a", "b", "a", "b", "a"],
        "score": [10, 9, 7, 9, 1, 3],
        "name": ["p", "q", "r", "s", "t", "u"],
    })
    expected = pd.DataFrame({"g": ["a", "a", "b", "b"], "score": [9, 9, 10, 7], "name": ["q", "s", "p", "r"]})
    _scenarios("top_n_per_group", [
        ("top 2 per group", lambda: f(df, "g", "score", 2), expected,
         "sort by group, then by score (high to low) with a stable sort so ties keep their order; groupby(...).head(n); reset_index(drop=True)"),
        ("n larger than a group", lambda: len(f(df, "g", "score", 10)), 6, None),
    ])


def check_enrich_orders(f):
    orders = pd.DataFrame({"order_id": [3, 1, 2, 4], "customer_id": ["c2", "c1", "c9", "c1"], "amount": [5.0, 7.0, 1.0, 2.0]})
    customers = pd.DataFrame({"customer_id": ["c1", "c2", "c3"], "country": ["Nepal", "India", "Japan"]})
    dup = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
    expected = pd.DataFrame({
        "order_id": [3, 1, 2, 4],
        "customer_id": ["c2", "c1", "c9", "c1"],
        "amount": [5.0, 7.0, 1.0, 2.0],
        "country": ["India", "Nepal", np.nan, "Nepal"],
        "customer_found": [True, True, False, True],
    })
    _scenarios("enrich_orders", [
        ("left join, original order, found flag", lambda: f(orders, customers), expected,
         "how='left' keeps every order; customer_found is True where the customer exists"),
        ("duplicate customers are rejected", lambda: f(orders, dup), Raises(ValueError),
         "validate='many_to_one' raises a MergeError, which is a kind of ValueError"),
    ])


def check_rfm_features(f):
    orders = pd.DataFrame({
        "customer_id": ["a", "a", "b", "a", "c", "b"],
        "order_date": pd.to_datetime(["2025-01-01", "2025-01-20", "2025-01-25", "2025-02-05", "2025-02-10", "2025-01-30"]),
        "amount": [10.0, 20.0, 5.0, 100.0, 50.0, 15.0],
    })
    cutoff = pd.Timestamp("2025-02-01")
    expected = pd.DataFrame(
        {"recency_days": [12, 2], "frequency": [2, 2], "monetary": [30.0, 20.0]},
        index=pd.Index(["a", "b"], name="customer_id"),
    )
    _scenarios("rfm_features", [
        ("only orders before the cutoff count", lambda: f(orders, cutoff), expected,
         "filter order_date < cutoff first; orders on or after the cutoff are the future and must not leak in"),
    ])


def check_monthly_pivot(f):
    orders = pd.DataFrame({
        "customer_id": ["b", "a", "a", "b", "a"],
        "order_date": pd.to_datetime(["2025-01-15", "2025-01-02", "2025-01-30", "2025-04-01", "2025-04-20"]),
        "amount": [5.0, 10.0, 2.5, 7.0, 1.0],
    })
    expected = pd.DataFrame(
        {"2025-01": [12.5, 5.0], "2025-02": [0.0, 0.0], "2025-03": [0.0, 0.0], "2025-04": [1.0, 7.0]},
        index=pd.Index(["a", "b"], name="customer_id"),
    )
    expected.columns.name = "month"
    _scenarios("monthly_pivot", [
        ("every month from first to last, filled with 0", lambda: f(orders), expected,
         "months with no orders at all (Feb, Mar) still need columns: reindex with every month in the range"),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `load_scores`

In [2]:
import pandas as pd


def load_scores(path) -> pd.DataFrame:
    return pd.read_csv(
        path,
        dtype={"id": str},
        parse_dates=["date"],
        na_values=["?", "-"],
    )

In [3]:
check_load_scores(load_scores)

PASS  all 4 tests passed for load_scores


Three arguments, three problems solved:

- `dtype={"id": str}` stops pandas from guessing that `007` is the number 7. Codes like IDs, ZIP codes and phone numbers look like numbers but aren't: you never do arithmetic on them, and the leading zeros matter.
- `parse_dates=["date"]` converts the column to `datetime64`, so you can subtract dates, extract months and filter by date ranges.
- `na_values=["?", "-"]` adds to the strings pandas already treats as missing (`NA`, `NaN`, `null`, the empty string and a few more). Without it, the whole `score` column would be read as text, because `?` isn't a number.

It's worth being explicit like this for any file you'll load more than once. The guessing is convenient for a first look, but if a later version of the file has a `?` in a column that used to be all numbers, the guessed type silently changes, and code further down breaks in confusing ways.

### Exercise 2: `missing_summary`

In [4]:
import pandas as pd


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean(),
    })
    summary = summary.rename_axis("column").reset_index()
    summary = summary.sort_values(["n_missing", "column"], ascending=[False, True])
    return summary.set_index("column").rename_axis(None)

In [5]:
check_missing_summary(missing_summary)

PASS  all 2 tests passed for missing_summary


`df.isna()` is a table of `True`/`False`. Its `sum()` counts the missing values per column, and its `mean()` gives the fraction, since `True` counts as 1. Both come back indexed by column name, so building the DataFrame from them lines everything up automatically.

The sorting is the only fiddly part. `sort_values` sorts by columns, not by the index, so the solution briefly moves the column names out of the index into a real column, sorts by both keys at once (count high to low, name A to Z), then puts them back. `rename_axis(None)` removes the temporary name from the index at the end.

Compare this with the Module 5 version: the counting is one expression here instead of a loop, and the result is a table you can sort, filter or plot.

### Exercise 3: `filter_and_sort`

In [6]:
import pandas as pd


def filter_and_sort(df: pd.DataFrame, species: str, min_mass: float) -> pd.DataFrame:
    mask = (df["species"] == species) & (df["body_mass_g"] >= min_mass)
    return (
        df.loc[mask, ["species", "island", "body_mass_g"]]
        .sort_values("body_mass_g", ascending=False)
        .reset_index(drop=True)
    )

In [7]:
check_filter_and_sort(filter_and_sort)

PASS  all 4 tests passed for filter_and_sort


`df.loc[mask, columns]` selects rows and columns in one step, which is the recommended way to take a subset.

Rows with a missing mass are excluded automatically: `NaN >= min_mass` is `False`, so they never pass the mask. That's usually what you want here, but remember it's happening, as the lesson's `size_class` example showed.

`reset_index(drop=True)` replaces the old row labels with `0, 1, 2, ...`. Without `drop=True`, the old labels would be kept as a new column called `index`. Resetting matters after filtering and sorting: otherwise `result.loc[0]` refers to whichever row had label 0 in the original table, which may not even be in the result, while `result.iloc[0]` is the first row. A fresh index makes the two agree again.

Every step returns a new DataFrame, so `df` is never modified.

### Exercise 4: `add_features`

In [8]:
import pandas as pd


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(
        bill_ratio=df["bill_length_mm"] / df["bill_depth_mm"],
        mass_kg=df["body_mass_g"] / 1000,
    )

In [9]:
check_add_features(add_features)

PASS  all 5 tests passed for add_features


`assign` returns a copy with the new columns added, so the caller's DataFrame is untouched. Writing `df["mass_kg"] = ...` inside the function would add the column to the *caller's* DataFrame, a side effect that's easy to miss: run the function twice and nothing seems wrong, but code elsewhere suddenly finds an extra column.

Division with a missing value gives a missing value, so no special handling is needed.

Ratios like `bill_ratio` are a classic piece of **feature engineering**: combining raw measurements into something closer to what actually distinguishes the classes. Here, Adelies have short, deep bills and the other two species have long, shallower ones, so the ratio separates them better than either measurement alone.

## Medium

### Exercise 5: `species_table`

In [10]:
import pandas as pd


def species_table(df: pd.DataFrame) -> pd.DataFrame:
    is_male = (df["sex"] == "male").where(df["sex"].notna())
    table = df.assign(is_male=is_male).groupby("species").agg(
        n=("species", "size"),
        mean_mass=("body_mass_g", "mean"),
        mean_flipper=("flipper_length_mm", "mean"),
        pct_male=("is_male", "mean"),
    )
    table[["mean_mass", "mean_flipper"]] = table[["mean_mass", "mean_flipper"]].round(1)
    return table.sort_values("mean_mass", ascending=False)

In [11]:
check_species_table(species_table)

PASS  all 1 tests passed for species_table


The interesting part is `pct_male`. The naive version, the mean of `df["sex"] == "male"`, counts penguins with unknown sex as "not male", which biases the fraction down. `.where(df["sex"].notna())` turns those entries into `NaN`, and since `mean` skips `NaN`, the fraction is computed over known values only. For the Gentoos in the check, that's 1 male out of 2 known, not 1 out of 3.

Adding `is_male` as a temporary column with `assign` lets it take part in the same named aggregation as everything else.

`n=("species", "size")` counts rows, including those with missing values elsewhere. `"count"` would count non-missing values of a specific column instead, which is a different question. Both have their uses, so pick deliberately.

Rounding only the two mean columns keeps `pct_male` precise. Rounding is for presentation. If these numbers feed into further calculations, round at the very end, or not at all.

### Exercise 6: `fill_by_group`

In [12]:
import pandas as pd


def fill_by_group(train: pd.DataFrame, test: pd.DataFrame, column: str, group: str):
    group_medians = train.groupby(group)[column].median()
    overall = train[column].median()

    def fill(df):
        fill_values = df[group].map(group_medians).fillna(overall)
        return df.assign(**{column: df[column].fillna(fill_values)})

    return fill(train), fill(test)

In [13]:
check_fill_by_group(fill_by_group)

PASS  all 3 tests passed for fill_by_group


The steps:

1. `train.groupby(group)[column].median()` gives one median per group, as a Series indexed by group name, computed from the training data only. A group with no values gets `NaN`.
2. `df[group].map(group_medians)` looks up each row's group median, giving a Series aligned with the rows. Groups that aren't in the training data, or whose median is `NaN`, come out as `NaN`...
3. ...which `.fillna(overall)` replaces with the overall training median.
4. `df[column].fillna(fill_values)` fills each missing value with the value for its own row. When you pass a Series to `fillna`, it's matched up by index label, row by row.

Using `map` with a Series of group statistics is an extremely common pandas pattern. It's a lookup table applied to a whole column at once, the dictionary lookup from Module 2, vectorized.

`assign(**{column: ...})` is needed because the column name is in a variable. `assign(column=...)` would create a column literally called `"column"`.

### Exercise 7: `zscore_within_group`

In [14]:
import numpy as np
import pandas as pd


def zscore_within_group(df: pd.DataFrame, column: str, group: str) -> pd.Series:
    grouped = df.groupby(group)[column]
    mean = grouped.transform("mean")
    std = grouped.transform("std", ddof=0)
    z = (df[column] - mean) / std.where(std > 0)
    return z.where(std > 0, 0.0)

In [15]:
check_zscore_within_group(zscore_within_group)

PASS  all 1 tests passed for zscore_within_group


`transform` is the key. Where `agg` returns one value per group, `transform` returns one value per *row*, with each row getting its own group's statistic. So `mean` and `std` have the same index as `df`, and the arithmetic lines up row by row.

The zero-std case is handled in two steps. `std.where(std > 0)` turns zero standard deviations into `NaN` before dividing, which avoids dividing by zero (and the warnings that come with it). Then `z.where(std > 0, 0.0)` puts 0 in exactly those rows.

`ddof=0` matters. pandas' `std` defaults to `ddof=1`, the *sample* standard deviation, unlike NumPy's default of 0. That's the same library mismatch flagged in Module 2's `mean_and_std` exercise. With `ddof=1`, a single-row group would give `NaN` rather than 0.

The result keeps `df`'s index, so you can add it straight back as a column: `df["mass_z"] = zscore_within_group(df, "body_mass_g", "species")`.

### Exercise 8: `one_hot_aligned`

In [16]:
import pandas as pd


def one_hot_aligned(train: pd.DataFrame, test: pd.DataFrame, columns: list[str]):
    train_enc = pd.get_dummies(train, columns=columns, dtype=int)
    test_enc = pd.get_dummies(test, columns=columns, dtype=int)
    test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)
    return train_enc, test_enc

In [17]:
check_one_hot_aligned(one_hot_aligned)

PASS  all 2 tests passed for one_hot_aligned


`reindex(columns=...)` does all the alignment in one step: it keeps the listed columns in that order, adds any that are missing (filled with `fill_value`), and drops any that aren't listed. Since the list comes from the training data, the test set ends up shaped exactly like the data the model was trained on.

What happens to a test penguin from an island the model never saw? All of its island columns are 0. That's a reasonable default, saying "none of the islands I know", but it's worth knowing it happens silently. In a real system you'd count how often it occurs, because a sudden rise in unseen categories usually means the incoming data has changed.

`dtype=int` gives 0/1 integers. Recent pandas versions default to `True`/`False` booleans, which work in most places but can surprise you in arithmetic or when saving to CSV.

scikit-learn's `OneHotEncoder(handle_unknown="ignore")` does the same thing, storing the categories during `fit`.

### Exercise 9: `top_n_per_group`

In [18]:
import pandas as pd


def top_n_per_group(df: pd.DataFrame, group: str, by: str, n: int) -> pd.DataFrame:
    ordered = df.sort_values([group, by], ascending=[True, False], kind="stable")
    return ordered.groupby(group).head(n).reset_index(drop=True)

In [19]:
check_top_n_per_group(top_n_per_group)

PASS  all 2 tests passed for top_n_per_group


Sort first, then take the first `n` rows of each group. `groupby(...).head(n)` keeps rows in their current order and returns the first `n` from each group, so the sorting does all the real work.

`kind="stable"` asks for a stable sort, which keeps rows with equal keys in their original order (the Module 2 idea). That's what makes ties predictable. It's a good habit whenever results need to be reproducible.

An alternative is `df.groupby(group, group_keys=False).apply(lambda g: g.nlargest(n, by))`, which reads naturally but calls a Python function once per group. Sorting once and using `head` runs as a few whole-table operations, which matters when there are thousands of groups.

## Hard

### Exercise 10: `enrich_orders`

In [20]:
import pandas as pd


def enrich_orders(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    merged = orders.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    merged["customer_found"] = merged["_merge"] == "both"
    merged = merged.drop(columns="_merge")
    assert len(merged) == len(orders)
    return merged

In [21]:
check_enrich_orders(enrich_orders)

PASS  all 2 tests passed for enrich_orders


Three arguments to `merge` do most of the work:

- `how="left"` keeps every order. The default, `inner`, would quietly drop the orders whose customer is missing, and with them some revenue.
- `validate="many_to_one"` checks that each customer appears at most once. If not, pandas raises a `MergeError`, which is a subclass of `ValueError`, so it satisfies the requirement without extra code. Without it, a duplicated customer would duplicate every one of their orders.
- `indicator=True` adds a `_merge` column saying whether each row was found in both tables. Turning it into a clean boolean column and dropping the original keeps the output tidy.

A left merge keeps the left table's row order, so the orders come back in their original order.

The `assert` at the end is cheap insurance: a check that the most important property of this function (one output row per order) actually holds. In a data pipeline, small assertions like this catch problems at the step where they happen, instead of three steps later in a confusing form.

### Exercise 11: `rfm_features`

In [22]:
import pandas as pd


def rfm_features(orders: pd.DataFrame, cutoff: pd.Timestamp) -> pd.DataFrame:
    past = orders[orders["order_date"] < cutoff]
    rfm = past.groupby("customer_id").agg(
        last_order=("order_date", "max"),
        frequency=("order_date", "size"),
        monetary=("amount", "sum"),
    )
    rfm["recency_days"] = (cutoff - rfm["last_order"]).dt.days
    return rfm[["recency_days", "frequency", "monetary"]].sort_index()

In [23]:
check_rfm_features(rfm_features)

PASS  all 1 tests passed for rfm_features


The filter comes **first**, before any aggregation. That ordering is the whole point. If you aggregated over all orders and filtered afterwards, a customer's "last order" might be one placed after the cutoff, and `recency_days` would come out negative, or quietly wrong.

`cutoff - rfm["last_order"]` subtracts every customer's last order date from the cutoff at once, giving durations, and `.dt.days` converts them to whole days.

Named aggregation computes all three statistics in one pass over the groups. `last_order` is a temporary column, used to compute recency and then dropped by selecting only the three columns we want.

These three numbers are surprisingly predictive. Customers who bought recently, often and a lot tend to keep buying, and a long gap since the last order is often the clearest sign that someone has stopped. The mini project builds on exactly this.

### Exercise 12: `monthly_pivot`

In [24]:
import pandas as pd


def monthly_pivot(orders: pd.DataFrame) -> pd.DataFrame:
    month = orders["order_date"].dt.to_period("M")
    table = (
        orders.assign(month=month)
        .pivot_table(index="customer_id", columns="month", values="amount", aggfunc="sum", fill_value=0.0)
    )
    all_months = pd.period_range(month.min(), month.max(), freq="M")
    table = table.reindex(columns=all_months, fill_value=0.0).sort_index()
    table.columns = table.columns.astype(str)
    table.columns.name = "month"
    return table

In [25]:
check_monthly_pivot(monthly_pivot)

PASS  all 1 tests passed for monthly_pivot


`.dt.to_period("M")` turns each date into a *period* representing its whole month. Periods sort correctly and can be turned into a complete range with `pd.period_range`, which is exactly what we need to fill in the gaps.

`pivot_table` does the reshaping: one row per customer, one column per month that has data, summing amounts where a customer ordered several times in the same month.

The trap is that `pivot_table` only creates columns for months that actually appear. In the check, nobody ordered in February or March, so those columns would simply be missing, and a model expecting twelve monthly columns would get ten, or get them shifted. Reindexing with the full range of months fixes that, filling the new columns with 0.

Converting the period labels to strings at the end (`"2025-01"`) makes the columns easy to work with and to save to CSV.

This is the same alignment idea as `one_hot_aligned`: decide on the full set of columns explicitly, instead of letting the data that happens to be present decide for you.

## What's next

The mini project in `04_mini_project.ipynb` predicts customer churn from raw order tables, and shows how easily the future leaks into features if you're not careful about dates.